# Projeto Final

### Preparo de dados

In [38]:
import os
from pathlib import Path
import numpy as np
import pandas as pd

# Caminho para o diretório principal dos dados
base_path = Path('./Dados_EEG/eegmmidb')


def tira_janelas(X, y, sub_processar, run_processar, base_path):
    if not base_path.is_dir():
        print(f"Erro: O diretório '{base_path}' não foi encontrado.")
        print(f"Diretório atual: '{os.getcwd()}'")
        return
    
    else:
        print(f"Iniciando processamento dos dados de: {base_path.resolve()}\n")

    # --- 3. LOOP PRINCIPAL PARA PROCESSAR OS ARQUIVOS ---
    for subject_id in sub_processar:
        for run_id in run_processar:
            
            s_id_str = f"{subject_id:03d}" # ex: 001
            r_id_str = f"{run_id:02d}"     # ex: 01

            ann_filename = f"SUB_{s_id_str}_ANN_{r_id_str}.csv"
            sig_filename = f"SUB_{s_id_str}_SIG_{r_id_str}.csv"

            ann_filepath = base_path / ann_filename
            sig_filepath = base_path / sig_filename

                # Verifica se ambos os arquivos (ANN e SIG) existem
            if ann_filepath.exists() and sig_filepath.exists():
                #print(f"Processando: Sujeito {s_id_str}, Run {r_id_str}")

                try:
                    # --- LEITURA DOS ARQUIVOS ---
                    colunas_ann = ['rotulo', 'duracao_s', 'tamanho_amostras', 'amostra_inicio', 'amostra_final']
                    
                    df_ann = pd.read_csv(ann_filepath, header=None, names=colunas_ann)

                    df_sig = pd.read_csv(sig_filepath, header=None)

                    # --- EXTRAÇÃO DOS SEGMENTOS E RÓTULOS ---
                    #
                    for index, row in df_ann.iterrows():
                        
                        rotulo = row['rotulo']
                        inicio = row['amostra_inicio']
                        fim = row['amostra_final']

                        inicio = int(inicio)-1 # python começa em zero
                        fim = int(fim)

                        segmento_sinal = df_sig.iloc[inicio:fim].values

                        X.append(segmento_sinal)
                        y.append(rotulo)

                except Exception as e:
                    print(f"  -> Erro ao processar o par de arquivos para S{s_id_str}/R{r_id_str}: {e}")
            else:
                # arquiv não encontrado
                print(f"Aviso: Par de arquivos para Sujeito {s_id_str}, Run {r_id_str} não encontrado. Pulando.")
                pass

    print("\n--- Processamento Concluído ---")
        
    # convt array NumPy
    y = np.array(y)

    print(f"Total de segmentos extraídos: {len(X)}")
    print(f"Total de rótulos extraídos: {len(y)}")
    print(f"Exemplo de um segmento (shape): {X[0].shape if X else 'N/A'}")
    print(f"Rótulos únicos encontrados: {np.unique(y) if y.size > 0 else 'N/A'}")

In [39]:
X_move = [] #segmento sinal
y_move = [] #rótulos

sub_move = range(1, 10) # 1 a 103
run_move = [1, 5, 9]    # 1 a 12

base_path = Path('./Dados_EEG/eegmmidb')

tira_janelas(X_move, y_move, sub_move, run_move, base_path)

Iniciando processamento dos dados de: C:\Users\eduar\TCC\TCC-EEG_Control\Programa_CNN\Dados_EEG\eegmmidb


--- Processamento Concluído ---
Total de segmentos extraídos: 810
Total de rótulos extraídos: 810
Exemplo de um segmento (shape): (672, 64)
Rótulos únicos encontrados: [1. 2. 3.]


In [40]:
X_imag = [] #segmento sinal
y_imag = [] #rótulos

sub_imag = range(1, 10) # 1 a 103
run_imag = [2, 6, 10]   # 1 a 12

base_path = Path('./Dados_EEG/eegmmidb')

tira_janelas(X_imag, y_imag, sub_imag, run_imag, base_path)

Iniciando processamento dos dados de: C:\Users\eduar\TCC\TCC-EEG_Control\Programa_CNN\Dados_EEG\eegmmidb


--- Processamento Concluído ---
Total de segmentos extraídos: 810
Total de rótulos extraídos: 810
Exemplo de um segmento (shape): (672, 64)
Rótulos únicos encontrados: [4. 5. 6.]


In [41]:
len(X_move[0]),len(X_imag[0])

(672, 672)

In [42]:
sinais = X_move

tamanhos_das_janelas = [janela.shape[0] for janela in sinais]

print("--- Verificação dos Tamanhos das Janelas ---")
print(f"Total de janelas extraídas: {len(tamanhos_das_janelas)}")

print("\n--- Resumo Estatístico dos Tamanhos ---")
print(f"Tamanho Mínimo: {np.min(tamanhos_das_janelas)} amostras")
print(f"Tamanho Máximo: {np.max(tamanhos_das_janelas)} amostras")

--- Verificação dos Tamanhos das Janelas ---
Total de janelas extraídas: 810

--- Resumo Estatístico dos Tamanhos ---
Tamanho Mínimo: 416 amostras
Tamanho Máximo: 672 amostras


Seleção Dados para utilizar

In [46]:
DADOS_X = X_move
DADOS_Y = y_move

# DADOS_X = X_move + X_imag
# DADOS_Y = y_move + y_imag

#### Data Augmentation (tvz use)

In [ ]:
# Parâmetro de ruído 
noise_factor = 0.01

# Parâmetro de escala
scale_factor = 0.1  

# Deslocar o sinal no tempo
def shift_signal(data, maDADOS_X_shift):
    shift = np.random.randint(-maDADOS_X_shift, maDADOS_X_shift)
    return np.roll(data, shift, aDADOS_Xis=0)

shift_maDADOS_X = 20  # Máximo deslocamento

from scipy.ndimage import gaussian_filter1d

# jitter, escalonamento, deslocamento e suavização
X_noise = DADOS_X + noise_factor * np.random.normal(loc=0.0, scale=1.0, size=DADOS_X.shape)
X_aug = X_noise * (1 + scale_factor * np.random.uniform(-1, 1, size=DADOS_X.shape))
X_aug = gaussian_filter1d(DADOS_X, sigma=2, aDADOS_Xis=1) 
X_aug = np.array([shift_signal(dado, shift_maDADOS_X) for dado in X_aug])

# Filtra rotulos a aumentar
# X_menos = DADOS_X[DADOS_Y != 0]
# Y_menos = DADOS_Y[DADOS_Y != 0]

# # data augmentation para equilibrar rotulos
# X_noise = X_menos + noise_factor * np.random.normal(loc=0.0, scale=1.0, size=X_menos.shape)
# X_aug = X_noise * (1 + scale_factor * np.random.uniform(-1, 1, size=X_noise.shape))
# X_aug = gaussian_filter1d(X_aug, sigma=2, aDADOS_Xis=1)
# X_aug = np.array([shift_signal(dado, shift_maDADOS_X) for dado in X_aug])


X_aug = np.concatenate([DADOS_X, X_aug], aDADOS_Xis=0)
y_aug = np.concatenate([DADOS_Y, Y_menos], aDADOS_Xis=0)

print(f"Forma dos dados: {X_aug.shape}")
print(f"Forma dos rótulos: {y_aug.shape}")

#### Separação Dados Treino e Validação

In [ ]:
from sklearn.model_selection import train_test_split
# DADOS_X = X_aug
# DADOS_Y = y_aug

X_train, X_test, y_train, y_test = train_test_split(DADOS_X, DADOS_Y, test_size=0.15, random_state=0)

print(len(X_train), len(y_train))
print(len(X_test), len(y_test))

In [44]:
import pickle

with open('dados_EEG.pkl', mode='wb') as f:
    pickle.dump([X_train, y_train, X_test, y_test], f)

### Importando Dados

In [45]:
with open('dados_EEG.pkl', mode='rb') as f:
    X_train, y_train, X_test, y_test = pickle.load(f)

print(len(X_train), len(y_train))
print(len(X_test), len(y_test))

688 688
122 122


# Arquitetura 1D CNN com Keras

In [ ]:
import tensorflow as tf
from keras.models import Sequential
from keras.layers import Conv1D, MaxPooling1D, Flatten, Dense, BatchNormalization, Input, Dropout
from keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam
"""
# Definindo o modelo
model = Sequential()

# Camada de entrada com Input(shape)
model.add(Input(shape=(656, qnt_canais)))

# 1ª Camada 
# Convolução 1D
model.add(Conv1D(filters=64, kernel_size=20, activation='relu'))
'''
# Batch Normalization
model.add(BatchNormalization())
'''
# 2ª Camada
# Convolução 1D
model.add(Conv1D(filters=64, kernel_size=20, activation='relu'))
# Batch Normalization
model.add(BatchNormalization())
# Dropout 50%
model.add(Dropout(0.5))
'''
# 3ª Camada
# Convolução 1D
model.add(Conv1D(filters=64, kernel_size=6, activation='relu'))
'''

# 4ª Camada
# Pooling 1D
model.add(MaxPooling1D(pool_size=2))

# 5ª Camada
# Convolução 1D
model.add(Conv1D(filters=64, kernel_size=6, activation='relu'))
# Dropout 50%
model.add(Dropout(0.5))

# 6ª Camada
# Flatten
model.add(Flatten())

# 7ª Camada
# Fully Conected
model.add(Dense(240, activation='relu'))
# Dropout 50%
model.add(Dropout(0.5))

# 8ª Camada
# Fully Conected
model.add(Dense(120, activation='relu' ))
# Dropout 50%
model.add(Dropout(0.5))
'''
# 9ª Camada
# Fully Conected
model.add(Dense(60, activation='relu'))
# Dropout 50%
model.add(Dropout(0.5))
'''

# 10ª Camada
# Fully Conected
model.add(Dense(3, activation='softmax'))  # Para classificação de múltiplas classes

# Compilando o modelo
model.compile(optimizer=Adam(learning_rate=0.0010), loss='sparse_categorical_crossentropy', metrics=['accuracy'])

# Resumo do modelo
model.summary()
"""

In [ ]:
# Definindo o modelo
model = Sequential()
qnt_canais = 1

# Camada de entrada com Input(shape)
model.add(Input(shape=(150, qnt_canais)))
'''
# Adicionando a camada de convolução 1D
model.add(Conv1D(filters=64, kernel_size=3, activation='relu'))
'''
# Convolução 1D
model.add(Conv1D(filters=10, kernel_size=6, activation='relu'))

# Dropout 50%
model.add(Dropout(0.5))

# Convolução 1D
model.add(Conv1D(filters=10, kernel_size=3, activation='relu'))

# Adicionando uma camada de pooling 1D
model.add(MaxPooling1D(pool_size=2))

# Flatten e camadas densas para classificação
model.add(Flatten())
model.add(Dense(80, activation='relu'))

# Dropout 50%
model.add(Dropout(0.5))
model.add(Dense(4, activation='softmax'))  # Para classificação de múltiplas classes

# Compilando o modelo
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

# Resumo do modelo
model.summary()

In [ ]:
# Dividindo treino/teste
from sklearn.model_selection import train_test_split

# Dados originais
#X_train, X_test, y_train, y_test = train_test_split(DADOS, ROTU, test_size=0.20, random_state=42)

# Dados aumentados
X_train, X_test, y_train, y_test = train_test_split(DADOS_aug, ROTU_aug, test_size=0.20, random_state=42)

# Define os callbacks
early_stopping = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.3, patience=2, min_lr=1e-6)

# Treina modelo
model.fit(X_train, y_train, 
          epochs=40, 
          batch_size=32, 
          validation_data=(X_test, y_test), 
          callbacks=[early_stopping, reduce_lr])  # callbacks


print(f"Tamanho do conjunto de treino: {X_train.shape[0]} amostras")
print(f"Tamanho do conjunto de teste: {X_test.shape[0]} amostras")

accuracy: 0.4434 - loss: 1.2756 - val_accuracy: 0.4200 - val_loss: 1.3564 - learning_rate: 9.0000e-05

In [ ]:
caminho_pesos = 'C:/Users/eduar/Sistemas Inteligentes/Projeto Final/Pesos_1Dcnn.weights.h5'

# Salvar os pesos do modelo
model.save_weights(caminho_pesos)

print(f"Pesos do modelo foram salvos em: {caminho_pesos}")

In [ ]:
# Caso já tenha seus dados de teste em X_test e y_test

# Normaliza os dados de teste da mesma forma que os dados de treinamento
X_test = X_test / np.max(X_test)

# Avaliar o modelo
loss, accuracy = model.evaluate(X_test, y_test)
print(f'Acurácia: {accuracy}')

# Fazer previsões
predictions = model.predict(X_test)


## Matriz Confusão

In [ ]:
# Precisa do predictions = model.predict(X_test) realizado a cima

# Converte as probabilidades em rótulos de classe
predicted_classes = np.argmax(predictions, axis=1)
#true_classes = np.argmax(y_test, axis=1)
# Se y_test for unidimensional, use-o diretamente
true_classes = y_test if len(y_test.shape) == 1 else np.argmax(y_test, axis=1)

# Calcula a matriz de confusão
conf_matrix = confusion_matrix(true_classes, predicted_classes)
print(conf_matrix)

# Define os rótulos das classes
class_names = ['Parado', 'DIreita', 'Esquerda', 'Frente']

# Cria uma plotagem da matriz de confusão
plt.figure(figsize=(7, 5))
sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
plt.xlabel('Previsto')
plt.ylabel('Real')
plt.title('Matrix de Confusão')
plt.savefig('confusion_matrix_1Dcnn.png')
plt.show()